# Programowanie baz danych — Tutorial
## AI przy pisaniu SQL — audyt odpowiedzi modelu

**Tryb:** Jupyter + JupySQL — wszystkie zapytania wykonywane na żywo
**SZBD:** PostgreSQL jako podstawowy, dopowiedzenia w **Oracle** i **SQL Server**
**Bazy:** Kotki + Kotki rozszerzone + Przykłady inne

---

## Plan zajęć

- Co mówią badania 2024–2026: liczby i taksonomia błędów generowanego SQL
- Budowa promptu: schemat z typami, ziarno, przykłady, standardy projektowe
- **Audyt czterech „odpowiedzi modelu”** (A–D) — zadanie centralne: znajdź błąd, udowodnij, popraw
- Proces czterech kroków: INTENT → MODEL → CHECK → VERIFY + lista kontrolna
- Pomost do optymalizacji: wynik poprawny — a ile kosztuje?
- Zadania 1–13 (części I–III)

**Jak pracować z notatnikiem:**
> 1. Uruchom pięć pierwszych komórek kodu — stanowi to bazę na całe zajęcia
>    (połączenia, schemat KOTKI, schemat Oracle, tabela `incydenty`).
> 2. W bloku AI-2 najpierw **samodzielnie** oceń „odpowiedź modelu” — zapisz
>    podejrzenia, a dopiero potem wykonaj zapytania kontrolne.
> 3. Odpowiedzi modeli są wpisane na stałe — notatnik jest w pełni wykonywalny
>    bez dostępu do zewnętrznego modelu LLM. Po zakończeniu analizy każdego przypadku
>    spróbuj uzyskać odpowiedź z rzeczywistego modelu językowego i poddaj ją temu
>    samemu audytowi.

In [3]:
from dotenv import load_dotenv

# Wczytanie sekretów (PG_CONN, MSSQL_CONN, ORACLE_CONN) z pliku .env
load_dotenv()

True

In [4]:
# JupySQL / SQLAlchemy — połączenia z trzema SZBD
%reload_ext sql

# Wczytanie connection stringów ze zmiennych środowiskowych (z pliku .env)
%sql $PG_CONN --alias pg
%sql $PG_CONN_KOTKI --alias pgkotki
%sql $MSSQL_CONN --alias mssql
%sql $ORACLE_CONN --alias oracle

%config SqlMagic.displaylimit = 100

Connecting and switching to connection 'pg'

Connecting and switching to connection 'pgkotki'

Connecting and switching to connection 'mssql'

Connecting and switching to connection 'oracle'

## KOTKI — wdrożenie w PostgreSQL (tabele TEMP)

Rozszerzony schemat PWr **KOTKI** wdrażamy — podobnie jak na poprzednich zajęciach — jako **tabele tymczasowe sesji** (`CREATE TEMP TABLE ...`). Publiczny schemat `chinook` na połączeniu `pg` traktujemy jako **wyłącznie do odczytu**.

## Laboratorium: 1 000 000 incydentów

Druga baza ćwiczeniowa: tabela **`incydenty`** (1 mln wierszy, konwencja KOTKI:
20 pseudonimów kotów, 12 wrogów, daty z okresu 10 lat) oraz `wrogowie_ref`
(12 wierszy — słownik wrogów). Zbiór ten posłuży w sekcji **EXPLAIN**, gdy po
weryfikacji poprawności przejdziemy do analizy kosztu wykonania.

Dwie celowe cechy generatora danych:

- `szkodliwosc` przyjmuje 1000 wartości (0–999), natomiast w przybliżeniu u co setnego wiersza
  (dokładnie dla `g % 97 = 0`, tj. w 10 309 rekordach) przyjmuje wartość `NULL`;
- `kraj` to wybór z siedmiu wartości: `'US'` dla co siódmego wiersza (`g % 7 = 0`),
  pozostałe to `PL`, `DE`, `FR`, `UK`, `ES` oraz `IT`.

Tabele są **tymczasowe (`TEMP`)** — istnieją wyłącznie w ramach bieżącej sesji. Bezpośrednio po ich utworzeniu wykonujemy
`ANALYZE` (aktualizacja statystyk dla optymalizatora zapytań). Tabele celowo **nie posiadają indeksów**.

In [ ]:
%%sql pgkotki
DROP TABLE IF EXISTS incydenty;
DROP TABLE IF EXISTS wrogowie_ref;

CREATE TEMP TABLE incydenty AS
SELECT 'KOT_' || ((g - 1) % 20)::text AS pseudo,
        'WROG_' || (((g - 1) / 20) % 12)::text AS imie_wroga,
       DATE '2015-01-01' + (g % 3650) AS data_incydentu,
    TIMESTAMP '2015-01-01'
    + (g % 3650) * INTERVAL '1 day'
    + (g % 24) * INTERVAL '1 hour' AS dataczas_incydentu,
       'KOC_' || g::text AS pseudo_id,
       g AS id,
       CASE WHEN g % 97 = 0 THEN NULL ELSE (g % 1000) END AS szkodliwosc,
       CASE WHEN g % 7 = 0 THEN 'US'
            ELSE (ARRAY['PL','DE','FR','UK','ES','IT'])[1 + g % 6] END AS kraj
FROM generate_series(1, 1000000) g;

CREATE TEMP TABLE wrogowie_ref AS
SELECT 'WROG_' || n::text AS imie_wroga, (n % 10) + 1 AS stopien
FROM generate_series(0, 11) n;

ANALYZE incydenty;
ANALYZE wrogowie_ref;

SELECT COUNT(*) AS incydentow,
       COUNT(szkodliwosc) AS z_szkodliwoscia,
       COUNT(DISTINCT szkodliwosc) AS wartosci_szkodliwosci
FROM incydenty

## Zanim użyjesz modelu: co mówią badania 2024–2026

Język SQL ma charakter deklaratywny: użytkownik określa, **co** chce uzyskać, natomiast optymalizator zapytań decyduje, **w jaki sposób** to zrealizować. Duży model językowy (LLM) dopasowuje wzorce na podstawie milionów przeanalizowanych zapytań — **nie ma jednak dostępu do zawartości bazy danych** i nie zna dokładnej intencji autora pytania.
Z tego względu wygenerowany kod SQL należy traktować jako **hipotezę**, a nie gotowe rozwiązanie. Wyniki badań z lat 2024–2026 są w tym zakresie zbieżne:

| Badanie | Główne wyniki |
|---|---|
| **Jin i in., PVLDB 19(5), 2026** — „Pervasive Annotation Errors Break Text-to-SQL Benchmarks and Leaderboards”: ręczny audyt **złotych zapytań** benchmarków BIRD Mini-Dev i Spider 2.0-Snow | **52,8%** przykładów w BIRD Mini-Dev i **62,8%** w Spider 2.0-Snow zawierało błędy adnotacji, obejmujące m.in. niezgodność zapytania referencyjnego z pytaniem, schematem bazy lub wiedzą dziedzinową. Błędy występują również we wzorcach referencyjnych, dlatego audyt przez wykonanie obowiązuje także wobec „oficjalnych” odpowiedzi. Autorzy wyróżniają cztery kategorie problemów: niezgodność logiki SQL z treścią pytania, niezgodność SQL z rzeczywistą strukturą lub zawartością bazy, błędy wynikające z braku wiedzy dziedzinowej oraz niejednoznaczność pytania lub oczekiwanego formatu wyniku. |
| **Shen i in., 2025** — taksonomia błędów wygenerowanego SQL (29 typów w 7 kategoriach), arXiv:2501.09310 | **37,3%** zapytań wygenerowanych przez LLM zawierało błąd; najczęstsze z nich to: agregacja niezgodna z pytaniem (`E5` w taksonomii autorów), niewłaściwy argument funkcji `COUNT` (`E6`), brak `DISTINCT` (`E8`) oraz niejawna konwersja typów (`C1`). Błędy semantyczne stanowiły największą z wyodrębnionych kategorii. Istotny udział miały także błędy formalne i strukturalne, a część przypadków wykraczała poza zaproponowaną taksonomię. Pięć analizowanych metod naprawczych zwiększało odsetek poprawnych zapytań, lecz jednocześnie wprowadzało nowe błędy i zwiększało opóźnienia. Nie należy zatem zakładać, że model zawsze skutecznie poprawi własny kod SQL na podstawie informacji zwrotnej z wykonania. |
| **Eizaguirre, Tissen, Sánchez-Artigas, EuroMLSys 2026** — „Both Ends Count! Just How Good are LLM Agents at Text-to-Big SQL?”: 1730 błędów z 930 błędnych tłumaczeń BIRD zaklasyfikowanych taksonomią Shena | Rozkład błędów: nieprawidłowy format wyniku **38,9%**, agregacja niezgodna z pytaniem **26,7%**, błędna projekcja **11,4%**, niewłaściwa tabela **8,6%**, błędny warunek **8,0%**, niejawna konwersja typów **4,6%**; czas wykonania zapytania TPC-H Q21 wzrósł **×13,3** przy skalowaniu SF10 → SF1000 na tym samym klastrze. Aż 93,16% błędów formatu wynikało z dodania nadmiarowych kolumn. Autorzy zaznaczają, że w ocenie Text-to-Big SQL wynik taki bywa częściowo użyteczny, dlatego nie należy traktować całych 38,9% na równi z krytycznymi błędami agregacji czy predykatów. |
| **Wang i in., 2018** (execution-guided decoding); **Mao i in., Findings ACL 2024**; **Zhai i in., Findings ACL 2025** (ExCoT) | Prace poświęcone wykorzystaniu informacji zwrotnej z wykonania zapytań: *execution-guided decoding* (odrzucanie lub korygowanie częściowo wygenerowanych programów na podstawie ich zachowania podczas wykonania), *DART-SQL* (łączenie przeformułowania pytania z iteracyjnym doskonaleniem SQL na podstawie zawartości bazy i wyników wykonania) oraz *ExCoT-DPO* (wykorzystanie dokładności wykonania do optymalizacji procesu rozumowania i generowania SQL). Podejście to polega na uruchamianiu wygenerowanego zapytania oraz jego **odrzucaniu lub poprawianiu na podstawie uzyskanych rezultatów**. |
| **Lin i in., PACMMOD 3(6), 2025** — „Are Your LLM-based Text-to-SQL Models Secure? Exploring SQL Injection via Backdoor Attacks” | Autorzy analizują ataki typu backdoor na modele Text-to-SQL, w których zatrute dane treningowe prowadzą do generowania złośliwych, wykonywalnych zapytań zawierających ładunki SQL injection. W eksperymencie zatrucie 0,44% danych pozwoliło osiągnąć skuteczność ataku na poziomie 79,41%. Badanie potwierdza zagrożenie wynikające z manipulacji zbiorami treningowymi (nie należy bez dodatkowych dowodów przenosić tych wartości bezpośrednio na przykłady *few-shot* w promptach). Wygenerowany SQL to **niezaufany kod**, dlatego należy stosować zasadę ograniczonego zaufania: konto tylko do odczytu, limity zasobów, ograniczenie dostępu do wybranych tabel i widoków, walidację składni oraz blokadę nieautoryzowanych operacji modyfikacji danych. |
| **Ascoli, Kandikonda i Choi (2025)** — „ETM: Modern Insights into Perspective on Text-to-SQL Evaluation in the Age of Large Language Models”, *Future Internet*, 17(8), 325. DOI: 10.3390/fi17080325 | Autorzy analizują ograniczenia metryk *Execution Accuracy* (EXE) oraz *Exact Set Matching* (ESM). EXE może uznać za poprawne zapytanie błędne strukturalnie, jeśli przypadkowo zwraca oczekiwany wynik, natomiast ESM może odrzucić zapytanie poprawne semantycznie o innej postaci składniowej. Zaproponowana metryka *Enhanced Tree Matching* (ETM), oparta na znormalizowanych drzewach składniowych (AST) i regułach równoważności, zredukowała odsetek błędnych ocen z 23,0% (EXE) i 28,9% (ESM) odpowiednio do 0,3% i 2,7%. Dowodzi to, że samo porównanie wyników wykonania nie stanowi wystarczającego kryterium oceny poprawności SQL. |
| **Zhong, Yu i Klein (2020)** — „Semantic Evaluation for Text-to-SQL with Distilled Test Suites”, *Proceedings of EMNLP 2020*, s. 396–411. DOI: 10.18653/v1/2020.emnlp-main.29 | Autorzy proponują metrykę *Test-Suite Accuracy*, polegającą na weryfikacji zapytań na zestawie odpowiednio przygotowanych wariantów bazy danych o wysokim pokryciu kodu. Pozwala to wykryć błędy logiczne niewidoczne w pojedynczej instancji danych. Wykazano, że tradycyjna metryka benchmarku Spider generowała średnio 2,5% (a maksymalnie 8,1%) fałszywie negatywnych ocen. Uzasadnia to konieczność testowania zapytań na zróżnicowanych przypadkach brzegowych (złączenia, agregacje, predykaty, obsługa `NULL`). |
| **Li, Gong, Kanellis i Narayanaswamy (2026)** — „Beyond Correctness: Resolving Underspecification in Agentic Text-to-SQL”, arXiv:2610.02739 | Autorzy analizują problem niedookreślenia intencji użytkownika w systemach agentowych Text-to-SQL. Agent może przyjąć nieuzgodnione założenia, które przypadkowo prowadzą do prawidłowego wyniku wykonania. Wprowadzony mechanizm *PlanPool* utrzymuje jawną listę pytań wymagających doprecyzowania. Eksperymenty na benchmarkach z rodzin BIRD-Interact i Spider potwierdzają poprawę wykrywania niejednoznaczności i ograniczenie nieuzasadnionych założeń. |

### Dodatkowe wyniki istotne dla interpretacji badań

Wyniki dotyczące dokładności wykonania wymagają ostrożnej interpretacji. Uzyskanie poprawnego rezultatu na pojedynczej instancji bazy danych nie dowodzi poprawności logicznej zapytania — dwie różne konstrukcje SQL mogą zwrócić identyczny wynik dla określonego stanu tabel, mimo że tylko jedna z nich wiernie realizuje specyfikację pytania.

W literaturze Text-to-SQL problem ten adresują dwa uzupełniające się podejścia:
- **Test-Suite Accuracy** (Zhong, Yu i Klein, 2020) — weryfikacja zapytania na wielu wariantach bazy danych w celu wykrycia błędów niewidocznych w pojedynczej próbce danych;
- **Enhanced Tree Matching** (Ascoli, Kandikonda i Choi, 2025) — ocena uwzględniająca strukturę składniową i semantykę zapytania jako uzupełnienie porównania wyników wykonania.

Ponadto Li i in. (2026) zwracają uwagę na problem niedookreślenia intencji użytkownika: nawet zgodny wynik wykonania nie gwarantuje, że model prawidłowo zinterpretował wszystkie założenia dziedzinowe, gdyż mógł przyjąć nieuzgodnioną interpretację przypadkowo prowadzącą do oczekiwanego rezultatu.

### Zasady weryfikacji SQL

Każde zapytanie wygenerowane przez model językowy lub pozyskane ze źródła zewnętrznego należy traktować jako hipotezę wymagającą weryfikacji. Zalecana procedura kontrolna obejmuje następujące etapy:
- **Weryfikacja interpretacji pytania** — określenie zakresu zwracanych informacji, poziomu szczegółowości (ziarna), ograniczeń oraz przyjętych założeń.
- **Weryfikacja struktury danych** — sprawdzenie tabel, kolumn, kluczy, relacji, więzów integralności oraz znaczenia biznesowego atrybutów.
- **Weryfikacja logiki zapytania** — analiza złączeń, predykatów, projekcji, funkcji agregujących, grupowania, eliminacji duplikatów oraz sortowania.
- **Weryfikacja przez wykonanie** — uruchomienie zapytania w bezpiecznym środowisku testowym i ocena zgodności rezultatu ze specyfikacją.
- **Weryfikacja niezależna** — skonstruowanie niezależnego zapytania kontrolnego, obliczenie wybranych wartości inną metodą lub sprawdzenie wyniku na przypadkach brzegowych.
- **Weryfikacja bezpieczeństwa i efektywności** — analiza planu wykonania, kosztu operacji, zużycia zasobów oraz wymaganych uprawnień.

Podczas porównywania wyników należy uwzględniać semantykę języka SQL: kolejność wierszy nie jest gwarantowana bez jawnej klauzuli `ORDER BY`, a istotny wpływ na rezultat mają duplikaty, wartości `NULL`, konwersje typów oraz reguły zaokrąglania (w szczególności zaokrąglanie do dwóch miejsc dziesiętnych ma uzasadnienie dla wartości pieniężnych, lecz nie dla dowolnych wielkości liczbowych).

> Każdą odpowiedź modelu językowego, każde wygenerowane zapytanie SQL oraz każde rozwiązanie zaczerpnięte ze źródeł zewnętrznych traktujemy jako hipotezę wymagającą weryfikacji.

### Wnioski praktyczne

1. **Błędy koncentrują się w warstwie semantycznej, a nie składniowej.** Poprawność składniowa i wykonanie zapytania bez komunikatów o błędach nie dowodzą zgodności z intencją pytania — zapytanie może realizować inną logikę lub zwracać wiarygodnie wyglądające, lecz zawyżone wartości (np. wskutek zwielokrotnienia wierszy przy złączeniu).
2. **Szczególnej kontroli wymagają agregacje, złączenia, obsługa `NULL` i format wyniku.** Należy weryfikować krotność relacji, poziom szczegółowości (ziarno), różnice między `COUNT(*)`, `COUNT(kolumna)` a `COUNT(DISTINCT kolumna)`, konwersje typów oraz granice przedziałów czasowych.
3. **Audyt przez wykonanie jest niskokosztowy i obowiązuje dwukierunkowo.** Badanie Jin i in. (2026) dowodzi, że błędy występują również w zapytaniach wzorcowych benchmarków, dlatego niezależne zapytanie kontrolne jest niezbędne także przy pracy z gotowymi referencjami.
4. **Nie należy bezwarunkowo polegać na automatycznej autokorekcie modelu.** Kolejne iteracje naprawcze mogą eliminować jeden defekt, wprowadzając jednocześnie inny błąd semantyczny.
5. **Pełna ocena wymaga połączenia analizy semantycznej, testów brzegowych oraz oceny bezpieczeństwa i wydajności.** Samo uruchomienie zapytania i porównanie wyników na jednej próbce danych stanowi wyłącznie jeden z etapów audytu, a nie jego jedyne kryterium. Ostatecznym miernikiem sukcesu jest uzyskanie merytorycznie poprawnej odpowiedzi na postawione pytanie przy zachowaniu bezpieczeństwa i akceptowalnego kosztu wykonania.

## AI-1. Zanim zapytasz: konstrukcja kontekstu wejściowego (promptu)

Model LLM generuje kod SQL **wyłącznie na podstawie dostarczonego kontekstu wejściowego** — nie ma dostępu do zawartości bazy danych. Skuteczny prompt powinien zawierać cztery kluczowe elementy:

### 1. Schemat z typami danych i relacjami — zamiast ogólnego opisu tabel

Model powinien otrzymać pełną definicję struktury: nazwy kolumn **wraz z typami danych**, klucze główne i obce, **krotność relacji (`1:N`, `M:N`)** oraz określenie ziarna tabeli (semantyki pojedynczego wiersza).

**Nieprecyzyjnie:** *„Podaj sumę przydziałów myszy kotów, które mają wrogów o stopniu wrogości co najmniej 7.”*

**Poprawnie (kontekst = DDL + ziarno):**

```text
Tabele (PostgreSQL):
  Kocury(imie, plec, pseudo PK, funkcja FK, szef FK→Kocury.pseudo,
         w_stadku_od, przydzial_myszy, myszy_extra, nr_bandy FK)
    ziarno: jeden wiersz = jeden kot (identyfikator: pseudo)
  Wrogowie(imie_wroga PK, stopien_wrogosci, gatunek, lapowka)
    ziarno: jeden wiersz = jeden wróg
  Wrogowie_kocurow(pseudo FK, imie_wroga FK, data_incydentu, opis_incydentu)
    ziarno: jeden wiersz = JEDEN incydent (para kot–wróg może się powtarzać!)
Pytanie: suma przydziałów myszy kotów mających wrogów o stopniu >= 7.
```

Zwróć uwagę na ostatnią linię opisu struktury: **ziarno tabeli incydentów** to informacja niemożliwa do wydedukowania z samych nazw kolumn — a to ona rozstrzyga, czy sumowanie przydziałów doprowadzi do duplikacji wierszy (zob. przypadek B oraz Zadanie 2).

### 2. Przykłady (few-shot) — preferencje stylistyczne i standardy projektowe

Dwa lub trzy przykłady par *pytanie → SQL* przekazują modelowi lokalne konwencje: sposób aliasowania, stosowanie literałów daty (np. `DATE '2026-06-30'`) zamiast funkcji `CURRENT_DATE` w raportach czy użycie `COALESCE` dla kolumn dopuszczających `NULL`. Badania (DAIL-SQL, PVLDB 2024; DIN-SQL, NeurIPS 2023) wykazują, że **sposób reprezentacji schematu w prompcie** oraz dekompozycja pytania mają większy wpływ na poprawność niż drobne modyfikacje stylistyczne.

### 3. Reguły ograniczające (guardrails)

Zwięzła lista rygorystycznych zasad projektowych, np.:

- wyniki raportowe weryfikujemy **zawsze z kontrolą liczności wierszy** (`COUNT(*)`) jako osobnym zapytaniem kontrolnym;
- **zakaz stosowania** `SELECT *` w zapytaniach raportowych — wszystkie kolumny należy wymienić jawnie;
- data raportu jest **ustalona na stałe** („na dzień X”), bez użycia `CURRENT_DATE` — zapewnia to determinizm i powtarzalność wyników;
- na dużych tabelach wykonanie zapytania poprzedzamy analizą planu (`EXPLAIN`).

### 4. Żądanie kodu SQL wraz z deklaracją ziarna wyniku

Należy poprosić model o wygenerowanie zapytania SQL **oraz** jednego zdania określającego semantykę wiersza wynikowego: *„Co reprezentuje pojedynczy wiersz zwracanego wyniku?”*. Taka deklaracja jest znacznie tańsza w weryfikacji niż pełny audyt złożonego zapytania i często natychmiast ujawnia błędną interpretację pytania (zob. „audyt zewnętrznego kodu SQL” w sekcji AI-2).

## Taksonomia błędów E1–E8 (lista kontrolna audytu)

Taksonomia Shen i in. (2025) obejmuje 29 szczegółowych typów błędów. Na potrzeby niniejszego warsztatu grupujemy je w **osiem klas (`E1`–`E8`)**, uporządkowanych według częstości występowania w badaniach Eizaguirre i in. (2026) *(uwaga: przyjęte oznaczenia `E1`–`E8` stanowią autorską klasyfikację warsztatową i nie są tożsame z oryginalnymi kodami z publikacji Shen i in., 2025)*. W części praktycznej przeprowadzamy pełny audyt czterech przypadków (**A–D**), reprezentujących klasy **E1–E4**.

| Kod | Klasa błędu | Typowe objawy | Podstawowy test kontrolny |
| --- | --- | --- | --- |
| **E1** | Granice i inkluzja | `BETWEEN` zamiast przedziału półotwartego `>=` / `<`; `>` zamiast `>=`; błędne granice przedziałów; pominięcie wartości brzegowych | Sprawdź rekordy leżące dokładnie na granicach warunku oraz bezpośrednio przed i za nimi |
| **E2** | Agregacja i złączenia (fan-out) | `SUM` lub `AVG` po `JOIN` z tabelą szczegółów (pozycji); `COUNT(*)` zamiast `COUNT(DISTINCT ...)`; zwielokrotnienie rekordów; błędny poziom agregacji | Porównaj liczbę rekordów przed i po operacji `JOIN`; zweryfikuj agregację na poziomie pojedynczej encji |
| **E3** | Konwersje typów i operacje na datach | Dzielenie całkowitoliczbowe zamiast zmiennoprzecinkowego; niejawne rzutowanie (`CAST`); błędne obliczanie wieku przez odejmowanie lat kalendarzowych; nieprawidłowe porównywanie dat i znaczników czasu | Sprawdź przypadki brzegowe: rocznice/urodziny, przełom roku, wartości `NULL`, dzielenie liczb całkowitych |
| **E4** | Niejednoznaczność i błędna interpretacja wymagań | Model przyjmuje nieuzgodnione założenia; myli pojęcia dziedzinowe; interpretuje pojęcia względne (np. *najlepszy*, *aktywny*) w sposób arbitralny | Sformułuj alternatywne interpretacje pytania i sprawdź, czy prowadzą do różnych zapytań lub wyników |
| **E5** | Agregacja per-grupa i logika podzapytań | Globalne `MAX` zamiast maksimum w obrębie grupy; błędna klauzula `GROUP BY`; niewłaściwe użycie `HAVING`; brak korelacji w podzapytaniu | Zweryfikuj wynik dla co najmniej dwóch grup o różnych wartościach ekstremalnych |
| **E6** | Projekcja i kompletność wyniku | Brak wymaganej kolumny; nadmiarowe kolumny; niewłaściwy poziom szczegółowości; brak `DISTINCT`; błędny alias | Porównaj zestaw kolumn wyniku ze specyfikacją pytania; sprawdź liczbę i unikatowość zwracanych rekordów |
| **E7** | Warunki i logika predykatów | Błędne operatory `AND` / `OR`; niewłaściwa negacja `NOT`; pominięcie logiki trójwartościowej (`NULL`); filtrowanie zamiast sortowania lub odwrotnie; błędna logika `EXISTS` | Przygotuj rekordy spełniające poszczególne warunki osobno i łącznie; przetestuj obsługę `NULL` |
| **E8** | Wybór tabel i relacji | Niewłaściwa tabela; błędny klucz złączenia; brak warunku złączenia (iloczyn kartezjański); niewłaściwy typ złączenia (`INNER` zamiast `LEFT`); pominięcie tabeli pośredniczącej | Zweryfikuj ścieżkę złączeń na diagramie ER; sprawdź rekordy bez dopasowania oraz z wieloma dopasowaniami |

Dwie najczęstsze kategorie błędów w literaturze — nieprawidłowy format wyniku (38,9%) oraz agregacja niezgodna z pytaniem (26,7%) — odpowiadają w praktyce klasom **E6/E4** oraz **E2/E5**. Błędy klas **E1** i **E3** występują rzadziej, lecz są szczególnie trudne do wykrycia, ponieważ zwracają wynik zbliżony do oczekiwanego.

> **Zasada zajęć:** każda odpowiedź modelu podlega **audytowi przez wykonanie** — uruchamiamy zarówno zapytanie wygenerowane przez model, jak i niezależne zapytanie kontrolne, a uzyskane rezultaty muszą być w pełni zgodne.

## AI-2. Audyt czterech „odpowiedzi modelu”

Poniżej przedstawiono cztery przykładowe odpowiedzi modelu — każda zawiera **dokładnie jeden błąd** (odpowiednio z klas E1–E4). Wszystkie zapytania są poprawne składniowo i wykonują się bez błędów, zwracając pozornie wiarygodne wyniki. Zalecana procedura pracy (w parach):

1. Przeczytaj treść pytania oraz odpowiedź modelu. **Samodzielnie** sformułuj hipotezę dotyczącą potencjalnego błędu — jeszcze przed uruchomieniem kodu.
2. Wykonaj zapytanie modelu i zapisz uzyskany wynik.
3. Wykonaj zapytanie(-a) kontrolne. **Wyniki muszą być identyczne** — każda rozbieżność stanowi dowód występowania błędu.
4. Opracuj poprawioną wersję zapytania i zweryfikuj ją przez wykonanie. Dopiero po tym kroku zapoznaj się z omówieniem.

> **Uwaga metodyczna:** zapytania przypisane modelowi są zapisane na stałe w notatniku — dostęp do zewnętrznego modelu LLM nie jest wymagany do realizacji ćwiczenia. Celem bloku jest doskonalenie umiejętności **analizy i audytu zewnętrznego kodu SQL**. Po przeanalizowaniu każdego przypadku warto wygenerować odpowiedź z rzeczywistego modelu językowego (stosując zasady z sekcji AI-1) i poddać ją tej samej procedurze audytu.

### Przypadek A (E1 — granice i inkluzja) — KOTKI

> **Pytanie do modelu:** *Które koty przyjęto do stada **od 1 grudnia 2008**?
> Podaj pseudonim i datę przyjęcia.*

**Odpowiedź modelu:**

```sql
SELECT pseudo, w_stadku_od
FROM Kocury
WHERE w_stadku_od > DATE '2008-12-01'
ORDER BY w_stadku_od
```

Treść pytania wskazuje granicę „od 1 grudnia 2008”, natomiast w warunku SQL użyto operatora ostrej nierówności `>` (co odpowiada dacie od `2008-12-02`).
Czy na samej granicy przedziału znajdują się jakieś rekordy? Przed uruchomieniem zapytań kontrolnych zastanów się: skoro sformułowanie „od dnia X” w języku naturalnym oznacza przedział domknięty lewostronnie, na czym polega pułapka tego zapisu?

In [35]:
%%sql pgkotki
-- odpowiedź modelu (E1: '>' zamiast '>=' — gubi wiersze dokładnie na granicy)
SELECT pseudo, w_stadku_od
FROM Kocury
WHERE w_stadku_od > DATE '2008-12-01'
ORDER BY w_stadku_od

6 rows affected.

pseudo,w_stadku_od
RURA,2009-09-01
LOLA,2009-10-14
ZERO,2010-03-01
PUSZYSTA,2010-11-18
UCHO,2011-01-01
MALY,2011-05-15


In [ ]:
%%sql pgkotki
-- kontrola 1: czy na granicy warunku są wiersze? (jeśli tak — model je stracił)
SELECT pseudo, w_stadku_od
FROM Kocury
WHERE w_stadku_od = DATE '2008-12-01'

In [ ]:
%%sql pgkotki
-- kontrola 2: niezależne zapytanie kontrolne — „od ... włącznie” = >=
SELECT pseudo, w_stadku_od
FROM Kocury
WHERE w_stadku_od >= DATE '2008-12-01'
ORDER BY w_stadku_od

### Przypadek A — omówienie (po ćwiczeniu)

Zapytanie modelu zwróciło 6 kotów (począwszy od `RURA` z datą `2009-09-01`). Zapytanie kontrolne 1 dowodzi jednak, że **na samej granicy przedziału znajduje się `PLACEK`** (`2008-12-01`) — poprawny wynik obejmuje zatem **7 kotów** (kontrola 2), z rekordem `PLACEK` na pierwszej pozycji.

- Sformułowanie „od 1 grudnia” w języku naturalnym oznacza przedział **domknięty (`>=`)**, a nie ostro większy (`>`). Błąd ten nie jest widoczny bezpośrednio w zwróconej tabeli, ponieważ pominięty wiersz nie generuje żadnego ostrzeżenia.
- Weryfikacja jest niskokosztowa: należy **policzyć rekordy leżące dokładnie na granicy warunku** (`WHERE x = <granica>`).
- Analogiczny problem występuje przy górnej granicy przedziału: operator `BETWEEN a AND b` **domyka oba końce przedziału**. Jeżeli specyfikacja wymaga zakresu „do końca 2021 roku, z wyłączeniem 31 grudnia”, użycie `BETWEEN` będzie błędem — należy zastosować przedział półotwarty (`>= '2021-01-01' AND < '2021-12-31'`).

**Wniosek (E1):** błędy wartości brzegowych mają charakter „cichy” — wynik wygląda poprawnie, lecz brakuje w nim pojedynczych rekordów. Procedura kontrolna wymaga każdorazowego sprawdzenia danych **na granicy** warunku. Badanie Jin i in. (2026) potwierdza powszechność tego błędu również w złotych zapytaniach benchmarków (np. nieuzasadnione stosowanie domkniętego operatora `BETWEEN`).

### Przypadek B (E2 — agregacja po złączeniu / fan-out) — Chinook

> **Pytanie do modelu:** *Który klient wydał najwięcej? Podaj imię, nazwisko i sumę.*

**Odpowiedź modelu:**

```sql
SELECT c.first_name || ' ' || c.last_name AS klient,
       ROUND(SUM(i.total), 2) AS wydal
FROM invoice i
JOIN invoice_line l ON l.invoice_id = i.invoice_id     -- <-- zbędne złączenie zmieniające ziarno
JOIN customer c ON c.customer_id = i.customer_id
GROUP BY c.first_name || ' ' || c.last_name
ORDER BY wydal DESC
LIMIT 1
```

Uzyskany wynik: **Helena Holý, 502,62**. Przed uruchomieniem zapytań kontrolnych oceń rzęd wielkości tej kwoty: suma **wszystkich** faktur w bazie wynosi ok. 2328,60 — czy pojedynczy klient mógł wygenerować niemal 22% całkowitego przychodu sklepu?

In [30]:
%%sql pg
-- odpowiedź modelu (E2: fan-out złączenia z pozycjami faktur)
SELECT c.first_name || ' ' || c.last_name AS klient,
       ROUND(SUM(i.total), 2) AS wydal_model
FROM invoice i
JOIN invoice_line l ON l.invoice_id = i.invoice_id
JOIN customer c ON c.customer_id = i.customer_id
GROUP BY c.first_name || ' ' || c.last_name
ORDER BY wydal_model DESC
LIMIT 1

1 rows affected.

klient,wydal_model
Helena Holý,502.62


In [ ]:
%%sql pg
-- kontrola 1: ziarno! ile faktur, ile pozycji, średnia liczba pozycji na fakturę?
SELECT (SELECT COUNT(*) FROM invoice)      AS faktury,
       (SELECT COUNT(*) FROM invoice_line) AS pozycje,
       ROUND((SELECT COUNT(*) FROM invoice_line)::numeric
             / (SELECT COUNT(*) FROM invoice), 2) AS pozycji_na_fakture

In [ ]:
%%sql pg
-- kontrola 2: niezależne zapytanie kontrolne — bez invoice_line (agregacja po fakturach)
SELECT c.first_name || ' ' || c.last_name AS klient,
       ROUND(SUM(i.total), 2) AS wydal_kontrolnie
FROM invoice i
JOIN customer c ON c.customer_id = i.customer_id
GROUP BY c.first_name || ' ' || c.last_name
ORDER BY wydal_kontrolnie DESC
LIMIT 3

In [ ]:
%%sql pg
-- kontrola 3: różnica na jednym kliencie — dlaczego 502,62 vs 49,62?
-- Faktury Heleny Holý i liczba pozycji każdej z nich:
SELECT i.invoice_id, i.total, COUNT(l.invoice_line_id) AS pozycji
FROM invoice i
JOIN invoice_line l ON l.invoice_id = i.invoice_id
JOIN customer c ON c.customer_id = i.customer_id
WHERE c.first_name = 'Helena' AND c.last_name = 'Holý'
GROUP BY i.invoice_id, i.total
ORDER BY i.invoice_id

### Przypadek B — omówienie (po ćwiczeniu)

Model dołączył tabelę `invoice_line`, błędnie zakładając jej przydatność podczas sumowania wartości faktur. Skutki tej operacji są następujące:

- **Zmiana ziarna danych:** po złączeniu pojedyncza faktura jest powielana tyle razy, ile zawiera pozycji (kontrola 1: **412 faktur wobec 2240 pozycji**, co daje średnio **5,44** pozycji na fakturę).
- Suma wartości `i.total` po złączeniu wynosi **20 848,62** zamiast rzeczywistych **2 328,60** (niemal dziewięciokrotne zawyżenie). Zwrócona przez model kwota (**502,62**) to rzeczywiste wydatki Heleny Holý (**49,62**), pomnożone przez liczbę pozycji na poszczególnych jej fakturach (kontrola 3: siedem faktur tego klienta liczy odpowiednio 9, 2, 4, 6, 1, 2 oraz 14 pozycji).
- Kontrola 2 (agregacja bez tabeli `invoice_line`) zwraca poprawny ranking: Helena Holý (**49,62**), Richard Cunningham (**47,62**), Luis Rojas (**46,62**). W zależności od rozkładu pozycji na fakturach błąd ten może całkowicie zmienić lidera rankingu.

Jest to jeden z najpoważniejszych błędów semantycznych: agregacja niezgodna ze strukturą pytania (26,7% błędów w badaniu Eizaguirre i in., 2026). Zjawisko to określa się mianem **fan-out**: złączenie `1:N` zwielokrotnia wiersze tabeli nadrzędnej przed wykonaniem agregacji, wskutek czego funkcja agregująca sumuje zduplikowane wartości. W taksonomii Shen i in. (2025) odpowiada to kategoriom *unaligned aggregation structure* oraz *missing DISTINCT*.

**Wniosek (E2):** przed zaakceptowaniem wyniku należy zweryfikować **ziarno** — porównać liczbę rekordów (`COUNT(*)`) przed i po każdym złączeniu. Zasada ogólna: stosowanie funkcji `SUM` lub `AVG` po złączeniu `1:N` jest poprawne wyłącznie wtedy, gdy agregujemy atrybuty tabeli podrzędnej (np. wartości pozycji); sumowanie atrybutów nagłówka po dołączeniu jego pozycji niemal zawsze prowadzi do błędu.

> Zjawisko *fan-out* obniża nie tylko poprawność merytoryczną, lecz także wydajność — przetwarzanie zwielokrotnionych wierszy generuje zbędne obciążenie silnika bazodanowego, co zostanie zilustrowane w sekcji poświęconej instrukcji `EXPLAIN`.

### Przypadek C (E3 — wiek i pełne lata) — KOTKI

> **Pytanie do modelu:** *Które koty mają co najmniej 17 **pełnych lat**?
> Raport na dzień **2026-06-30**. Podaj pseudonim i pełne lata.*

**Odpowiedź modelu:**

```sql
SELECT pseudo,
       EXTRACT(YEAR FROM DATE '2026-06-30') - EXTRACT(YEAR FROM w_stadku_od) AS pelne_lata
FROM Kocury
WHERE EXTRACT(YEAR FROM DATE '2026-06-30') - EXTRACT(YEAR FROM w_stadku_od) >= 17
ORDER BY pseudo
```

*(Uwaga: w kontekście schematu KOTKI kolumna `w_stadku_od` oznacza datę przystąpienia do stada, co w zadaniu traktowane jest jako miara stażu/wieku w stadzie).*

Uruchom zapytanie i zweryfikuj dwa aspekty: (a) czy w zbiorze wynikowym nie znalazły się rekordy, dla których rocznica przypada **po** dniu raportu (30 czerwca), co oznacza, że ostatni pełny rok jeszcze nie upłynął; (b) czy obliczona wartość reprezentuje **pełne lata**, czy jedynie różnicę lat kalendarzowych.

In [ ]:
%%sql pgkotki
-- odpowiedź modelu (E3: różnica lat przez EXTRACT(YEAR) przekłamuje o 1)
SELECT pseudo,
       EXTRACT(YEAR FROM DATE '2026-06-30') - EXTRACT(YEAR FROM w_stadku_od) AS pelne_lata_model
FROM Kocury
WHERE EXTRACT(YEAR FROM DATE '2026-06-30') - EXTRACT(YEAR FROM w_stadku_od) >= 17
ORDER BY pseudo

In [ ]:
%%sql pgkotki
-- kontrola 1: pełne lata = pełne rocznice; wiek liczy AGE od dnia raportu
SELECT pseudo, w_stadku_od,
       EXTRACT(YEAR FROM AGE(DATE '2026-06-30', w_stadku_od)) AS pelne_lata_poprawnie
FROM Kocury
WHERE EXTRACT(YEAR FROM AGE(DATE '2026-06-30', w_stadku_od)) >= 17
ORDER BY pelne_lata_poprawnie DESC, pseudo

In [ ]:
%%sql pgkotki
-- kontrola 2: wektor wyników obu metod obok siebie — gdzie i o ile się różnią?
SELECT pseudo, w_stadku_od,
       EXTRACT(YEAR FROM DATE '2026-06-30') - EXTRACT(YEAR FROM w_stadku_od) AS naiwnie,
       EXTRACT(YEAR FROM AGE(DATE '2026-06-30', w_stadku_od)) AS pelne_lata
FROM Kocury
ORDER BY pelne_lata DESC, pseudo

### Przypadek C — omówienie (po ćwiczeniu)

Uproszczona różnica `EXTRACT(YEAR FROM d1) - EXTRACT(YEAR FROM d2)` oblicza różnicę **lat kalendarzowych (roczników)**, a nie **pełnych lat** — zalicza cały rok kalendarzowy nawet wówczas, gdy rocznica jeszcze nie nastąpiła. Dla daty raportu `2026-06-30`:

- **Model zwrócił 14 kotów**, podczas gdy poprawny wynik obejmuje **12 rekordów** — `LOLA` i `RURA` mają po **16** pełnych lat i nie spełniają warunku zadania (ich rocznice przypadają odpowiednio w październiku 2009 oraz wrześniu 2009).
- Wśród pozostałych 12 rekordów obliczona wartość jest zawyżona o 1 rok u siedmiu kotów (`DAMA`, `LYSY`, `MALA`, `MAN`, `PLACEK`, `RAFA`, `SZYBKA`; przykładowo dla rekordu `PLACEK` różnica roczników wynosi **18**, podczas gdy liczba pełnych lat to **17**, ponieważ rocznica przypada 1 grudnia), natomiast zgodna u pięciu (`BOLEK`, `KURKA`, `LASKA`, `TYGRYS`, `ZOMBI` — rocznice przed 30 czerwca).
- Oba objawy wynikają z tej samej przyczyny: metoda uproszczona i poprawna dają rozbieżne rezultaty **wyłącznie** dla rekordów, których rocznica wypada po dacie raportu. Dobór przypadku testowego powinien zatem celowo obejmować rekord z rocznicą po dniu raportowania.

Poprawna implementacja w PostgreSQL wykorzystuje funkcję `AGE`: `EXTRACT(YEAR FROM AGE(data_raportu, data_od))`, która oblicza pełne okresy roczne. Odpowiedniki w pozostałych analizowanych dialektach SQL:

- **Oracle:** `TRUNC(MONTHS_BETWEEN(DATE '2026-06-30', d) / 12)` — funkcja `MONTHS_BETWEEN` zwraca liczbę miesięcy wraz z częścią ułamkową, a dzielenie przez 12 i obcięcie części ułamkowej (`TRUNC`) wyznacza liczbę pełnych lat.
- **SQL Server:** funkcja `DATEDIFF(YEAR, d, @do)` zlicza **przekroczenia granic lat kalendarzowych** (analogicznie do metody uproszczonej); wyznaczenie pełnych lat wymaga korekty o dzień rocznicy: `DATEDIFF(YEAR, d, @do) - CASE WHEN DATEADD(YEAR, DATEDIFF(YEAR, d, @do), d) > @do THEN 1 ELSE 0 END`. Zagadnienia związane z obsługą dat i przedziałów czasowych rozwijają Zadanie 1 oraz Zadanie 8.

**Wniosek (E3):** błędy w arytmetyce dat mają charakter systemowy — poszczególne SZBD implementują funkcje czasu odmiennie, a konstrukcje oparte na `EXTRACT(YEAR)` lub `DATEDIFF(YEAR)` operują na latach kalendarzowych. Przy obliczaniu pełnych lat należy zawsze zweryfikować wynik na przypadku brzegowym, którego rocznica przypada **po dacie raportu**.

> **Wskazówka dotycząca promptów:** data raportu została ustalona na stałe (`2026-06-30`) zamiast użycia funkcji `CURRENT_DATE`, co gwarantuje pełną powtarzalność wyników niezależnie od terminu uruchomienia notatnika.

### Przypadek D (E4 — agregacja per-grupa) — Chinook

> **Pytanie do modelu:** *Dla każdego gatunku muzycznego znajdź jego
> **najdroższy** utwór. Podaj gatunek, tytuł utworu i cenę.*
> Jeśli najdroższych utworów w gatunku jest kilka, podaj dowolny z nich.

**Odpowiedź modelu:**

```sql
SELECT g.name AS gatunek, t.name AS utwor, t.unit_price AS cena
FROM track t
JOIN genre g ON g.genre_id = t.genre_id
WHERE t.unit_price = (SELECT MAX(unit_price) FROM track)
ORDER BY g.name, t.name
```

Model wyznaczył globalne maksimum `MAX(unit_price)` w podzapytaniu nieskorelowanym i porównał z nim ceny utworów ze **wszystkich** gatunków. Oszacuj przed uruchomieniem: ile gatunków powinno pojawić się w wyniku, jeśli w bazie Chinook 25 gatunków posiada przypisane utwory? Ile gatunków oraz ile wierszy zwróci zapytanie modelu?

In [ ]:
%%sql pg
-- odpowiedź modelu (E4: globalne MAX zamiast MAX per gatunek)
SELECT g.name AS gatunek, t.name AS utwor, t.unit_price AS cena
FROM track t
JOIN genre g ON g.genre_id = t.genre_id
WHERE t.unit_price = (SELECT MAX(unit_price) FROM track)
ORDER BY g.name, t.name

In [ ]:
%%sql pg
-- kontrola 1: ziarno pytania — ile gatunków w ogóle ma utwory?
-- (pytanie mówi "dla każdego gatunku": wynik ma mieć tyle wierszy, ile gatunków)
SELECT COUNT(DISTINCT genre_id) AS gatunki_z_utworami
FROM track

In [ ]:
%%sql pg
-- kontrola 2: niezależne zapytanie kontrolne — maksimum PER GATUNEK
-- (agregacja GROUP BY gatunku; remisy w gatunku rozstrzygamy niżej)
SELECT g.name AS gatunek, MAX(t.unit_price) AS max_cena
FROM track t
JOIN genre g ON g.genre_id = t.genre_id
GROUP BY g.name
ORDER BY g.name

### Przypadek D — omówienie (po ćwiczeniu)

- **Kontrola 1:** w bazie Chinook **25 gatunków posiada przypisane utwory** — dokładnie tyle wierszy powinien zawierać wynik końcowy (po jednym najdroższym utworze dla każdego gatunku).
- **Odpowiedź modelu** zwróciła **213 utworów należących zaledwie do 5 gatunków** (*Comedy*, *Drama*, *Sci Fi & Fantasy*, *Science Fiction*, *TV Shows*), ponieważ tylko w tych kategoriach występują utwory osiągające **globalne** maksimum ceny (`1,99`). Pozostałe 20 gatunków ma niższe ceny maksymalne (np. *Alternative*: `0,99`) i zostało całkowicie pominiętych w wyniku.
- **Kontrola 2** (agregacja `GROUP BY`) potwierdza, że **20 gatunków osiąga maksimum na poziomie `0,99`**, a jedynie 5 gatunków na poziomie **`1,99`**. Pełna realizacja specyfikacji pytania wymaga ponadto jednoznacznego rozstrzygnięcia remisów wewnątrz grup.

Błąd ten należy do klasy **E4: niejednoznaczność pytania / błędna interpretacja zakresu agregacji** (w taksonomii Shen i in., 2025 klasyfikowany jako *unaligned aggregation structure*): model zinterpretował predykat „najdroższy” jako cechę globalną całego zbioru zamiast właściwości lokalnej w obrębie grupy. Różnica między `MAX` globalnym a grupowym jest wyłącznie semantyczna — zapytanie wykonuje się poprawnie i zwraca wiarygodne dane, jednak **80% grup zostało bezśladowo odfiltrowanych**.

**Wniosek (E4):** jeżeli specyfikacja zawiera sformułowanie typu „dla każdej grupy X znajdź wartość ekstremalną”, należy zweryfikować **liczbę unikatowych grup** w wyniku (`COUNT(DISTINCT grupa)`). Musi być ona zgodna z liczbą grup w tabeli źródłowej; rozbieżność dowodzi błędnego zakresu agregacji.

**Rozstrzygnięcie remisów:** zapytanie kontrolne 2 wyznacza cenę maksymalną dla każdego gatunku, ale nie identyfikuje konkretnego utworu — w bazie Chinook wiele utworów w ramach jednego gatunku osiąga tę samą cenę maksymalną (`0,99` lub `1,99`). Ponieważ treść zadania dopuszcza wybór dowolnego z nich, właściwym rozwiązaniem jest zastosowanie funkcji okna `ROW_NUMBER() OVER (PARTITION BY genre_id ORDER BY unit_price DESC)` i odfiltrowanie rekordów warunkiem `WHERE rn = 1`, co gwarantuje zwrócenie dokładnie jednego utworu dla każdego z 25 gatunków.

## AI-3. Proces INTENT → MODEL → CHECK → VERIFY

Cztery przeanalizowane przypadki z bloku AI-2 odpowiadają klasom błędów, które można skutecznie zidentyfikować za pomocą **niskokosztowych zapytań kontrolnych**. Poniżej zestawiono te kroki w ustrukturyzowany proces weryfikacji.

### Krok 1 — INTENT: zdefiniuj oczekiwany wynik przed utworzeniem promptu

Przed sformułowaniem zapytania do modelu określ precyzyjnie: **co stanowi pojedynczy wiersz wyniku** (ziarno), jakie kolumny mają zostać zwrócone, jakie obowiązują warunki brzegowe oraz na jaki dzień generowany jest raport. Etap ten zajmuje kilkadziesiąt sekund, a pozwala wyeliminować większość błędów klas E2 i E4 dzięki wcześniejszemu określeniu oczekiwanej liczności zbioru wynikowego.

### Krok 2 — MODEL: skonstruuj prompt z pełnym kontekstem schematu (AI-1)

Przekaż pełne definicje DDL wraz z typami danych i relacjami, określ ziarno tabel, dodaj standardy projektowe oraz 1–2 przykłady wzorcowe. Zażądaj wygenerowania kodu SQL **oraz jednego zdania** opisującego semantykę pojedynczego wiersza wyniku.

### Krok 3 — CHECK: wykonaj zapytania kontrolne przed akceptacją wyniku

Minimalny zestaw testów dla każdego zapytania wykorzystującego agregacje lub złączenia:

| Kontrola | Wykrywany defekt | Koszt |
|---|---|---|
| `COUNT(*)` przed i po każdym złączeniu | efekt *fan-out* (E2) | 1 zapytanie |
| rekordy **na granicy** warunku (`WHERE x = <granica>`) | błędy inkluzji granic (E1) | 1 zapytanie |
| przypadek brzegowy (rekord z rocznicą po dacie raportu) | błędy w arytmetyce dat (E3) | 0 — weryfikacja analityczna |
| `COUNT(DISTINCT grupa)` wobec liczby grup w bazie | błędny zakres agregacji per-grupa (E4) | 1 zapytanie |
| niezależne zapytanie realizujące to samo zadanie inną metodą | wszystkie klasy błędów | 1 zapytanie |
| `EXPLAIN` / `EXPLAIN ANALYZE` | koszt wykonania, struktura planu | 0–1 wykonanie |

### Krok 4 — VERIFY: uruchom i porównaj rezultaty

Wyniki obu zapytań (modelu oraz kontrolnego) muszą być identyczne **co do liczby wierszy oraz wartości liczbowych**. Jakakolwiek rozbieżność stanowi **dowód błędu**. W takiej sytuacji należy skorygować zapytanie ręcznie lub powrócić do kroku 2, uzupełniając prompt o doprecyzowanie wymagań oraz informację zwrotną z wykonania — mechanizm ten (iteracyjne doskonalenie z wykorzystaniem wyników wykonania) opisują m.in. Wang i in. (2018), Mao i in. (2024) oraz Zhai i in. (2025).

## EXPLAIN — pomost do optymalizacji

Dotychczasowa analiza koncentrowała się na **poprawności merytorycznej** wyniku. Audyt przez wykonanie dostarcza jednak dodatkowej korzyści: uruchamiając zapytanie, można jednocześnie zweryfikować, **jakiego nakładu pracy wymaga ono od silnika bazodanowego**. Stanowi to wprowadzenie do zagadnień optymalizacji zapytań.

**Uwaga techniczna:** polecenie `EXPLAIN` (bez opcji `ANALYZE`) **nie wykonuje** zapytania — prezentuje wyłącznie plan szacowany przez optymalizator na podstawie statystyk. Polecenie `EXPLAIN ANALYZE` **rzeczywiście wykonuje** zapytanie i uzupełnia plan o rzeczywiste czasy oraz faktyczne liczności przetwarzanych wierszy (na dużych tabelach operacja ta może być czasochłonna; w przypadku instrukcji modyfikujących dane, takich jak `UPDATE` czy `DELETE`, należy stosować ją wyłącznie wewnątrz transakcji wycofywanej przez `ROLLBACK`).

In [ ]:
%%sql pg
-- EXPLAIN: plan szacowany — ile wierszy i jakim kosztem planer OBLICZA
-- porównajmy odpowiedź modelu (fan-out) z kontrolą 2 z przypadku B
EXPLAIN
SELECT c.first_name || ' ' || c.last_name AS klient,
       ROUND(SUM(i.total), 2)
FROM invoice i
JOIN invoice_line l ON l.invoice_id = i.invoice_id
JOIN customer c ON c.customer_id = i.customer_id
GROUP BY c.first_name || ' ' || c.last_name
ORDER BY 2 DESC
LIMIT 1

In [ ]:
%%sql pg
-- ten sam wynik, ale bez zbędnego JOIN invoice_line (kontrola 2 z B):
-- plan krótszy, sumuje 412 faktur zamiast 2240 pozycji
EXPLAIN
SELECT c.first_name || ' ' || c.last_name AS klient,
       ROUND(SUM(i.total), 2)
FROM invoice i
JOIN customer c ON c.customer_id = i.customer_id
GROUP BY c.first_name || ' ' || c.last_name
ORDER BY 2 DESC
LIMIT 1

W małej bazie Chinook różnice w koszcie wykonania są pomijalne (rzędu setek wierszy), jednak sam mechanizm jest wyraźnie widoczny: zbędne złączenie z tabelą pozycji **zwielokrotnia** 412 faktur do **2240 wierszy** na wejściu do operatora agregacji (zob. parametr `rows=` w węźle `Hash Join`), podczas gdy zapytanie poprawne agreguje bezpośrednio 412 rekordów. W przypadku tabel liczących miliony wierszy analogiczny błąd generuje koszt o rząd wielkości wyższy. Szczegółowa analiza operatorów planu (`Seq Scan`, selektywność predykatów, zarządzanie buforami) zostanie omówiona na zajęciach poświęconych optymalizacji.

W skali systemów produkcyjnych Eizaguirre i in. (2026) wykazali na benchmarku TPC-H, że czas wykonania zapytania Q21 wzrósł **×13,3** przy zwiększeniu skali danych z SF10 do SF1000 na tym samym klastrze — koszt nadmiarowych operacji rośnie wraz z wolumenem danych. Ponadto, jak wykazali Lin i in. (2025), wygenerowany kod SQL może zawierać ukryte podatności (backdoory), dlatego **niezależnie od weryfikacji poprawności** należy uruchamiać go na koncie z uprawnieniami wyłącznie do odczytu oraz z skonfigurowanym limitem czasu wykonania (`statement_timeout`).

In [ ]:
%%sql pg
-- na koniec: 1 mln incydentów — EXPLAIN ANALYZE wykonuje zapytanie
-- i pokazuje rzeczywiste liczności; filtr kraj='US' przepuszcza ~1/7 wierszy
EXPLAIN (ANALYZE, BUFFERS, TIMING OFF, SUMMARY ON)
SELECT COUNT(*) FROM incydenty WHERE kraj = 'US'

Plan pełnego skanowania tabeli: operator `Seq Scan` odczytuje 1 mln wierszy, filtr predykatu przepuszcza **~143 tysiące** rekordów (`actual rows=142857`), odrzucając pozostałe (`Rows Removed by Filter: 857143`). Szacunek optymalizatora (`rows=` w węźle `Seq Scan`, rzędu 144–146 tysięcy) może nieznacznie wahać się między uruchomieniami, ponieważ statystyki polecenia `ANALYZE` opierają się na losowej próbce danych; wartość `actual rows` pozostaje natomiast deterministyczna. Statystyka buforów `local read=9408` określa liczbę stron tabeli odczytanych z pamięci masowej. Analizę planu wykonania należy prowadzić od liści drzewa w górę: identyfikując sposób odczytu tabel, liczbę przepuszczanych rekordów na każdym etapie oraz węzły generujące największy koszt.

## Z. Zadania 

Celem ćwiczeń jest zbadanie, czy model językowy potrafi poprawnie przekształcić wymagania sformułowane w języku naturalnym na zapytania SQL w dialekcie PostgreSQL.

W trakcie realizacji zadań należy zwrócić szczególną uwagę na następujące aspekty:

- prawidłową interpretację treści polecenia;
- dobór właściwych tabel oraz relacji;
- poprawność predykatów i operatorów logicznych;
- właściwe zastosowanie funkcji agregujących i grupowania;
- obsługę wartości `NULL`;
- eliminację błędów wynikających z nadmiarowych lub nieprawidłowych złączeń;
- poprawność zapytań dla wartości brzegowych;
- kompletność i zgodność strukturalną wyników;
- wydajność oraz plan wykonania zapytań.

## Zasady wykonywania zadań

Dla każdego zadania należy wykonać następujące kroki:

1. Sformułować prompt dla wybranego modelu językowego.
2. Pozyskać wygenerowane zapytanie SQL w dialekcie PostgreSQL.
3. Zweryfikować poprawność składniową zapytania.
4. Uruchomić zapytanie i ocenić zgodność wyniku ze specyfikacją zadania.
5. Skonstruować niezależne zapytanie kontrolne.
6. Zidentyfikować ewentualne defekty i przyporządkować je do odpowiedniej klasy błędów.
7. Opracować poprawioną wersję zapytania wraz z uzasadnieniem wprowadzonych zmian.

Nie należy uznawać zapytania za poprawne wyłącznie na podstawie faktu jego bezbłędnego wykonania.

> Zadania należy realizować w parach.

# Część I. Mały zbiór danych — KOTKI

W tej części wykorzystaj schemat zawierający informacje o kotach, ich funkcjach, bandach oraz wrogach.

### Zadanie 1. Poprawność warunków i przedziałów dat

**Polecenie dla modelu:**

Wyświetl koty, które dołączyły do stada w 2008 roku. Wynik powinien zawierać pseudonim kota, imię, funkcję oraz datę dołączenia.

**Sprawdź:**

- czy model poprawnie określił zakres dat;
- czy uwzględnił pełny rok kalendarzowy;
- czy zastosował konstrukcję odporną na występowanie składowej czasowej (godzinowej) w kolumnie daty.

**Dodatkowe pytanie:** Czy zastosowanie operatora `BETWEEN` zawsze zapewnia poprawny wynik przy filtrowaniu według roku? Przygotuj przypadek testowy ilustrujący różnicę.

### Zadanie 2. Agregacje i zwielokrotnienie wierszy

**Polecenie dla modelu:**

Oblicz sumę przydziału myszy dla kotów, które miały przynajmniej jeden incydent z wrogiem.

**Sprawdź:**

- czy model poprawnie łączy tabele `kocury` i `wrogowie_kocurow`;
- czy suma przydziału myszy nie została zawyżona wskutek wielokrotnego występowania tego samego kota w tabeli incydentów (efekt *fan-out*);
- czy model potrafi zaproponować rozwiązanie wykorzystujące predykat `EXISTS`.

**Dodatkowe pytanie:** Dlaczego bezpośrednie złączenie tabel i zastosowanie funkcji `SUM()` prowadzi w tym przypadku do błędnego wyniku?

### Zadanie 3. Agregacja w ramach grupy

**Polecenie dla modelu:**

Dla każdej bandy znajdź kota lub koty, które mają największy przydział myszy w tej bandzie. Jeżeli kilka kotów ma ten sam maksymalny przydział, należy wyświetlić wszystkie.

Wynik powinien zawierać nazwę bandy, pseudonim kota i przydział myszy.

**Sprawdź:**

- czy model oblicza maksimum oddzielnie dla każdej bandy;
- czy nie stosuje maksimum globalnego;
- czy prawidłowo obsługuje remisy;
- czy uwzględnia bandy, które nie posiadają żadnych członków.

### Zadanie 4. Obsługa wartości NULL i kompletność wyników

**Polecenie dla modelu (część A):**

Wyświetl wszystkich wrogów, którzy nie otrzymali łapówki.

**Polecenie dla modelu (część B):**

Wyświetl wszystkie koty wraz z liczbą incydentów, w których uczestniczyły. Koty bez incydentów również powinny znaleźć się w wyniku.

**Sprawdź:**

- czy model stosuje operator `IS NULL` zamiast błędnego porównania `= NULL`;
- czy prawidłowo dobiera rodzaj złączenia (`LEFT JOIN`);
- czy zlicza rzeczywiste incydenty (`COUNT(kolumna)`), a nie liczbę wierszy powstałych po złączeniu (`COUNT(*)`);
- czy koty bez incydentów otrzymują wartość `0`.

**Dodatkowe pytanie:** Jaka jest różnica pomiędzy `COUNT(*)` a `COUNT(kolumna)` przy zastosowaniu `LEFT JOIN`?

### Zadanie 5. Niejednoznaczność wymagań

**Polecenie dla modelu:**

Znajdź najbardziej zaangażowanego kota.

Najpierw sprawdź, czy model wygeneruje zapytanie bez zadawania pytań doprecyzowujących. Następnie przeanalizuj, czy specyfikacja jest wystarczająco precyzyjna.

Przygotuj co najmniej trzy alternatywne interpretacje pojęcia „najbardziej zaangażowany”, np.:

- kot z największą łączną liczbą incydentów;
- kot, który miał incydenty z największą liczbą unikatowych wrogów;
- kot, który uczestniczył w incydentach z wrogami o największej łącznej szkodliwości lub stopniu wrogości.

**Zadanie dodatkowe:** Przeformułuj polecenie w taki sposób, aby posiadało jedną, ścisłą interpretację prowadzącą do jednoznacznego zapytania SQL.

---

# Część II. Duży zbiór danych — INCYDENTY

W tej części wykorzystaj tabele tymczasowe `incydenty` i `wrogowie_ref`, utworzone za pomocą dostarczonego skryptu PostgreSQL.

Tabela `incydenty` zawiera milion rekordów. Dane obejmują 20 kotów, 12 wrogów, datę i znacznik czasu incydentu, kraj oraz wartość szkodliwości, która może być nieokreślona (`NULL`).

### Zadanie 6. Różne znaczenia funkcji COUNT

**Polecenie dla modelu:**

Przygotuj raport zawierający:

- liczbę wszystkich incydentów;
- liczbę incydentów, dla których określono szkodliwość;
- liczbę unikatowych wartości szkodliwości;
- liczbę unikatowych kotów;
- liczbę unikatowych wrogów;
- liczbę unikatowych par kot–wróg.

**Sprawdź:**

- czy model rozróżnia działanie `COUNT(*)`, `COUNT(kolumna)` oraz `COUNT(DISTINCT kolumna)`;
- czy poprawnie zlicza unikatowe pary dwóch kolumn;
- czy prawidłowo traktuje wartości `NULL`.

**Dodatkowe pytanie:** Dlaczego liczba wszystkich incydentów nie musi być równa liczbie unikatowych par kot–wróg?

### Zadanie 7. NULL i zapytania antyzłączeniowe

**Polecenie dla modelu:**

Znajdź liczby całkowite z przedziału od 1000 do 1010, które nie występują jako wartości szkodliwości żadnego incydentu.

Wykorzystaj w tym celu funkcję `generate_series()` oraz dane z tabeli `incydenty`.

**Sprawdź:**

- czy model rozumie semantykę operatora `NOT IN` w obecności wartości `NULL`;
- czy zabezpiecza podzapytanie przed zwracaniem wartości `NULL`;
- czy potrafi zaproponować równoważne rozwiązanie z wykorzystaniem operatora `NOT EXISTS`.

**Dodatkowe pytanie:** Dlaczego zapytanie z klauzulą `NOT IN (SELECT szkodliwosc FROM incydenty)` zwraca pusty zbiór wyników zamiast oczekiwanego zakresu liczb?

### Zadanie 8. Daty i znaczenie części godzinowej

**Polecenie dla modelu:**

Policz liczbę incydentów, które wystąpiły w 2020 roku. Wynik przedstaw w dwóch wariantach:

- wykorzystując kolumnę `data_incydentu` (typ `DATE`);
- wykorzystując kolumnę `dataczas_incydentu` (typ `TIMESTAMP`).

Następnie porównaj oba uzyskane rezultaty.

**Sprawdź:**

- czy model rozróżnia typy `DATE` oraz `TIMESTAMP`;
- czy prawidłowo wyznacza początek i koniec przedziału czasowego;
- czy nie pomija incydentów z ostatniego dnia roku z niezerową godziną;
- czy stosuje przedział półotwarty (`>=` oraz `<`).

**Dodatkowe pytanie:** Przygotuj zapytanie identyfikujące incydenty z ostatniego dnia roku (`31 grudnia`), które wystąpiły po godzinie `00:00:00`.

### Zadanie 9. Agregacje i grupowanie

**Polecenie dla modelu:**

Dla każdego kota przygotuj zestawienie obejmujące:

- łączną liczbę incydentów;
- liczbę incydentów z określoną szkodliwością;
- średnią szkodliwość;
- minimalną i maksymalną szkodliwość.

Posortuj wynik malejąco według łącznej liczby incydentów.

**Sprawdź:**

- czy model prawidłowo stosuje klauzulę `GROUP BY`;
- czy uwzględnia fakt, że funkcje `AVG`, `MIN` i `MAX` ignorują wartości `NULL`;
- czy rozróżnia liczbę wszystkich incydentów od liczby niepustych wartości szkodliwości;
- czy poprawnie definiuje kierunek sortowania.

**Dodatkowe pytanie:** Czy średnia szkodliwość obliczana przez funkcję `AVG()` jest średnią ze wszystkich incydentów, czy wyłącznie z tych rekordów, dla których szkodliwość została określona?

### Zadanie 10. Filtrowanie po agregacji

**Polecenie dla modelu:**

Znajdź koty, które uczestniczyły w więcej niż 40 000 incydentów. Dla każdego z tych kotów podaj łączną liczbę incydentów oraz średnią szkodliwość.

**Sprawdź:**

- czy model poprawnie rozróżnia klauzule `WHERE` oraz `HAVING`;
- czy filtruje grupy po agregacji, a nie pojedyncze rekordy przed agregacją;
- czy potrafi wyjaśnić logiczną kolejność przetwarzania klauzul w zapytaniu SQL.

**Dodatkowe pytanie:** Zmodyfikuj zapytanie tak, aby uwzględniało wyłącznie koty, dla których liczba incydentów przekracza 40 000, a jednocześnie średnia szkodliwość jest wyższa od zadanego progu.

### Zadanie 11. Złączenia i poprawność agregacji

**Polecenie dla modelu:**

Przygotuj raport prezentujący dla każdego stopnia wrogości:

- łączną liczbę incydentów;
- liczbę unikatowych kotów uczestniczących w tych incydentach;
- średnią szkodliwość.

Wykorzystaj tabele `incydenty` oraz `wrogowie_ref`.

**Sprawdź:**

- czy model łączy tabele po właściwym kluczu (`imie_wroga`);
- czy odróżnia łączną liczbę incydentów od liczby unikatowych kotów (`COUNT(DISTINCT pseudo)`);
- czy prawidłowo grupuje dane według stopnia wrogości;
- czy nie stosuje nieuzasadnionego słowa kluczowego `DISTINCT` w innych agregatach.

**Dodatkowe pytanie:** Jakie konsekwencje dla wyniku będzie miało połączenie tabel po kolumnie, która nie stanowi klucza unikatowego?

### Zadanie 12. Wydajność i analiza planu wykonania

**Polecenie dla modelu:**

Wyznacz liczbę incydentów kota `KOT_7`, które wystąpiły w latach 2018–2020.
Zrealizuj to zadanie w dwóch wariantach:

1. filtrując dane przed agregacją z użyciem klauzuli `WHERE`;
2. grupując dane i filtrując je po agregacji z użyciem klauzuli `GROUP BY` oraz `HAVING`.

Przeanalizuj plany wykonania obu zapytań z wykorzystaniem polecenia:

```sql
EXPLAIN (ANALYZE, BUFFERS)
```

**Sprawdź:**

- czy model prawidłowo interpretuje operatory planu wykonania oraz statystyki buforów;
- czy potrafi wskazać różnicę w liczbie przetwarzanych wierszy oraz koszcie wykonania między filtrowaniem przed agregacją (`WHERE`) a filtrowaniem po agregacji (`GROUP BY ... HAVING`).

---

# Część III. Zadanie przekrojowe — krytyczna ocena odpowiedzi modelu

### Zadanie 13. Model jako autor i recenzent kodu SQL

Wybierz trzy zadania z poprzednich części. Dla każdego z nich wykonaj następującą procedurę:

1. Poproś model o wygenerowanie zapytania SQL.
2. Poproś ten sam model o krytyczną ocenę własnego rozwiązania.
3. Poproś o wskazanie potencjalnych błędów logicznych oraz przypadków brzegowych.
4. Przeprowadź niezależną weryfikację zapytania w bazie danych.
5. Opracuj ostateczną, poprawioną wersję zapytania.

W dokumentacji wynikowej zamieść:

- treść polecenia;
- pierwotnie wygenerowane zapytanie;
- autokrytykę modelu;
- zidentyfikowane błędy;
- poprawione zapytanie;
- opis procedury weryfikacji;
- klasyfikację błędu według taksonomii E1–E8.

## Proponowana forma dokumentacji

Dla każdego analizowanego zadania przygotuj zestawienie w formie poniższej tabeli:

| Element | Opis |
|---|---|
| Polecenie | Treść zadania w języku naturalnym |
| Model | Nazwa modelu i zastosowana konfiguracja |
| SQL wygenerowany | Pierwotna odpowiedź modelu |
| Poprawność składniowa | Tak / Nie |
| Poprawność logiczna | Tak / Nie / Częściowo |
| Zgodność z poleceniem | Tak / Nie / Częściowo |
| Wykryty błąd | Szczegółowy opis defektu |
| Kategoria błędu | Klasyfikacja (E1–E8) |
| Weryfikacja | Opis zapytania kontrolnego i metody sprawdzenia |
| SQL poprawiony | Ostateczna, zweryfikowana wersja zapytania |
| Wnioski | Zwięzłe podsumowanie analizy |